In [25]:
import pandas as pd
import numpy as np
import joblib
import mlflow
import mlflow.sklearn
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

In [2]:
version = 2
df = pd.read_parquet(f"../data/processed/client_month_features_{version}.parquet")

df = df.sort_values(['client_id', 'year_month']).copy()

In [64]:
path = "../models"

In [3]:
train = df[df['year_month'] < '2022-12'].copy()
val   = df[df['year_month'] == '2022-12'].copy()

In [4]:
lag = [col for col in df.columns if col.startswith("lag")]
rolling = [col for col in df.columns if "rolling" in col]
cumulative = [
    'amount_11_cumsum', 'tx_11_cumsum', 'tx_total_cumsum',
    'amount_11_cummean', 'amount_11_hist_max',
    'amount_11_hist_min', 'amount_11_hist_mean',
    'amount_11_hist_std', 'tx_11_share_cum'
]
other = [
    'amount_11_pct_change_lag', 'amount_11_abs_change_lag',
    'amount_11_vs_3m_avg_lag', 'amount_11_vs_cummean_lag',
    'amount_11_vs_hist_max_lag', 'amount_11_vs_hist_min_lag',
    'amount_11_hist_cv_lag', 'amount_11_hist_range_lag',
    'amount_11_is_spike_lag', 'amount_11_is_dip_lag',
    'amount_11_consecutive_growth_lag',
    'amount_11_is_zero_lag', 'amount_11_active_ratio_lag'
]
season = ['month', 'quarter', 'amount_11_same_month_last_year',
          'is_quarter_start', 'is_quarter_end']
event = [col for col in df.columns if "type" in col]

In [5]:
def preprocess(df, features):
    df = df.copy()
    
    lag_cols = [c for c in features if c.startswith("lag")]
    roll_cols = [c for c in features if "rolling" in c]
    
    df[lag_cols] = df[lag_cols].fillna(0)
    
    for col in roll_cols:
        df[col] = df[col].fillna(df[col].mean())
    
    df[features] = df[features].fillna(0)
    
    return df

In [6]:
def compute_global_metrics(y_true, y_pred):
    return {
        "global_rmse": np.sqrt(mean_squared_error(y_true, y_pred)),
        "global_mae": mean_absolute_error(y_true, y_pred),
        "global_r2": r2_score(y_true, y_pred),
    }

In [15]:
def compute_segment_diagnostics(y_true, y_pred, n_segments=5):
    tmp = pd.DataFrame({"true": y_true, "pred": y_pred})
    tmp["segment"] = pd.qcut(tmp["true"], q=n_segments)
    
    metrics = {}
    
    for seg in tmp["segment"].unique():
        part = tmp[tmp["segment"] == seg]
        rmse = np.sqrt(mean_squared_error(part["true"], part["pred"]))
        safe_name = str(seg)
        safe_name = safe_name.replace("(", "") \
                            .replace(")", "") \
                            .replace(",", "_") \
                            .replace("]", "") \
                            .replace("[", "")

        metrics[f"segment_{safe_name}_rmse"] = rmse    
    return metrics

In [65]:
import joblib
import numpy as np
from lightgbm import LGBMRegressor
import mlflow
import os

mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("client_amount_forecast_v2")


def run_experiment(
    feature_set_name,
    features,
    log_target=False,
    segment_thresholds=None,       # список квантилей для сегментов по lag1_amount_11_sum
    log_target_per_segment=None    # список True/False для каждого сегмента
):
    """
    feature_set_name: имя набора признаков
    features: список признаков
    log_target: логарифмировать целевую переменную для обычной модели
    segment_thresholds: квантильные пороги для сегментов (например [0.75, 0.95])
    log_target_per_segment: список True/False для каждого сегмента, если None используется log_target
    """
    with mlflow.start_run(run_name=feature_set_name):
        mlflow.log_param("dataset_version", version)
        mlflow.log_param("feature_set", feature_set_name)
        mlflow.log_param("log_target", log_target)
        mlflow.log_param("n_features", len(features))
        mlflow.log_param("segment_thresholds", segment_thresholds)
        mlflow.log_param("log_target_per_segment", log_target_per_segment)

        # Препроцессинг
        train_p = preprocess(train, features)
        val_p   = preprocess(val, features)

        X_train = train_p[features]
        y_train = train_p["amount_11_sum"]
        X_val   = val_p[features]
        y_val   = val_p["amount_11_sum"]

        preds = np.zeros(len(val_p))

        if segment_thresholds is not None:
            # Сегментация по lag1_amount_11_sum
            lag_train = train_p["lag1_amount_11_sum"]
            lag_val   = val_p["lag1_amount_11_sum"]
            thresholds = np.quantile(lag_train, segment_thresholds)

            def assign_segment(x):
                for i, t in enumerate(thresholds):
                    if x <= t:
                        return i
                return len(thresholds)  # верхний сегмент

            train_seg = lag_train.apply(assign_segment)
            val_seg   = lag_val.apply(assign_segment)
            n_segments = len(thresholds) + 1

            # Если log_target_per_segment не список нужной длины, используем обычный log_target для всех сегментов
            if log_target_per_segment is None or len(log_target_per_segment) != n_segments:
                log_target_per_segment = [log_target] * n_segments

            for s in np.unique(train_seg):
                mask_tr = train_seg == s
                mask_val = val_seg == s

                if mask_tr.sum() == 0 or mask_val.sum() == 0:
                    continue

                model_s = LGBMRegressor(n_estimators=500, verbose=-1)

                # Используем логарифмирование для конкретного сегмента
                if log_target_per_segment[s]:
                    model_s.fit(X_train[mask_tr], np.log1p(y_train[mask_tr]))
                    preds[mask_val] = np.expm1(model_s.predict(X_val[mask_val]))
                else:
                    model_s.fit(X_train[mask_tr], y_train[mask_tr])
                    preds[mask_val] = model_s.predict(X_val[mask_val])

                model_name = f"{path}/{feature_set_name}_seg{s}.pkl"
                joblib.dump(model_s, model_name)
                mlflow.log_artifact(model_name)

        else:
            # Обычная модель без сегментации
            model = LGBMRegressor(n_estimators=500, verbose=-1)
            if log_target:
                model.fit(X_train, np.log1p(y_train))
                preds = np.expm1(model.predict(X_val))
            else:
                model.fit(X_train, y_train)
                preds = model.predict(X_val)

            model_name = f"{path}/{feature_set_name}.pkl"
            joblib.dump(model, model_name)
            mlflow.log_artifact(model_name)

        # Глобальные метрики
        metrics = compute_global_metrics(y_val, preds)
        mlflow.log_metrics(metrics)

        # Сегментные метрики для диагностики
        seg_metrics = compute_segment_diagnostics(y_val, preds)
        mlflow.log_metrics(seg_metrics)

        print("\nRESULT:")
        for k, v in metrics.items():
            print(f"{k}: {v:.2f}")

        return metrics["global_rmse"]

In [36]:
features_lag = lag

run_experiment(
    feature_set_name="lag_only_raw",
    features=features_lag,
    log_target=False
)

run_experiment(
    feature_set_name="lag_only_log",
    features=features_lag,
    log_target=True
)


RESULT:
global_rmse: 449174307.23
global_mae: 26282396.53
global_r2: 0.15
🏃 View run lag_only_raw at: http://127.0.0.1:5000/#/experiments/2/runs/1fd47e0b6b8d47cc8af3d5f8c38985b0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2

RESULT:
global_rmse: 467368964.93
global_mae: 15044391.88
global_r2: 0.08
🏃 View run lag_only_log at: http://127.0.0.1:5000/#/experiments/2/runs/50c3eda262884fbdba213da85a043bc6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


np.float64(467368964.92569554)

In [37]:
features_lr = lag + rolling

run_experiment(
    feature_set_name="lag_rolling_raw",
    features=features_lr,
    log_target=False
)


RESULT:
global_rmse: 583754579.24
global_mae: 26497880.54
global_r2: -0.43
🏃 View run lag_rolling_raw at: http://127.0.0.1:5000/#/experiments/2/runs/ec13e345d1344b5eb097671342c25f76
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


np.float64(583754579.2394242)

In [18]:
print(df[rolling].isna().mean().sort_values(ascending=False).head())

amount_11_sum_rolling_std_6m     0.108131
amount_11_sum_rolling_std_3m     0.108131
amount_11_sum_rolling_std_12m    0.108131
amount_11_sum_rolling_mean_3m    0.054910
tx_11_count_rolling_mean_3m      0.054910
dtype: float64


In [20]:
df[['amount_11_sum','lag1_amount_11_sum','amount_11_sum_rolling_mean_3m']].head(10)

,amount_11_sum,lag1_amount_11_sum,amount_11_sum_rolling_mean_3m
0,2.521474e+04,NaN,NaN
1,1.132841e+05,2.521474e+04,25214.736328
2,1.568723e+04,1.132841e+05,69249.399414
3,6.125297e+04,1.568723e+04,51395.343424
4,1.007069e+05,6.125297e+04,63408.088867
5,1.256567e+05,1.007069e+05,59215.698242
6,2.637390e+04,1.256567e+05,95872.183594
7,4.324907e+05,NaN,NaN
8,1.359269e+06,4.324907e+05,432490.687500
9,4.539212e+05,1.359269e+06,895879.968750


In [ ]:
# Финальный набор признаков без rolling
features_final = lag + other + season + event

run_experiment(
    feature_set_name="final_features_raw",
    features=features_final,
    log_target=False
)



RESULT:
global_rmse: 405041550.00
global_mae: 24739817.10
global_r2: 0.31
🏃 View run final_features_raw at: http://127.0.0.1:5000/#/experiments/2/runs/8287b48e833e4beb9d0c1d7eb970e085
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


np.float64(405041549.9995937)

In [39]:
run_experiment(
    feature_set_name="final_features_log",
    features=features_final,
    log_target=True
)


RESULT:
global_rmse: 449611484.86
global_mae: 14598018.24
global_r2: 0.15
🏃 View run final_features_log at: http://127.0.0.1:5000/#/experiments/2/runs/42e79f500c5c4e0d82c8c8295b9e20f7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


np.float64(449611484.86443675)

In [45]:
rmse_final_segmented = run_experiment(
    feature_set_name="final_features_segmented_3",
    features=features_final,
    log_target=False,
    segment_thresholds=[0.95],   # включаем сегментацию
)


RESULT:
global_rmse: 385338002.69
global_mae: 21770897.08
global_r2: 0.38
🏃 View run final_features_segmented_3 at: http://127.0.0.1:5000/#/experiments/2/runs/460a8a7488c546a08024b67e72272c6a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [51]:
rmse_final_segmented = run_experiment(
    feature_set_name="final_features_segmented_3",
    features=features_final,
    log_target=False,
    segment_thresholds=[0.99],   # включаем сегментацию
)


RESULT:
global_rmse: 398122770.01
global_mae: 22386007.57
global_r2: 0.34
🏃 View run final_features_segmented_3 at: http://127.0.0.1:5000/#/experiments/2/runs/a9e0ef88f618439f84a1a26f7bf70aa8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [54]:
rmse = run_experiment(
    feature_set_name="final_features_segmented_custom_log",
    features=features_final,
    segment_thresholds=[0.95],       # 0–95%, 95+%
    log_target_per_segment=[True, True]  # только верхний сегмент логарифмируем
)


RESULT:
global_rmse: 394350874.42
global_mae: 13046898.22
global_r2: 0.35
🏃 View run final_features_segmented_custom_log at: http://127.0.0.1:5000/#/experiments/2/runs/6935249db2e54f19ac85e67d071de709
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [55]:
rmse = run_experiment(
    feature_set_name="final_features_segmented_custom_log",
    features=features_final,
    segment_thresholds=[0.95],       # 0–95%, 95+%
    log_target_per_segment=[False, True]  # только верхний сегмент логарифмируем
)


RESULT:
global_rmse: 398445440.16
global_mae: 19752994.48
global_r2: 0.33
🏃 View run final_features_segmented_custom_log at: http://127.0.0.1:5000/#/experiments/2/runs/bb6aa950e5684000921f31266cf050cf
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [66]:
rmse = run_experiment(
    feature_set_name="final_features_segmented_custom_log",
    features=features_final,
    segment_thresholds=[0.95],       # 0–95%, 95+%
    log_target_per_segment=[True, False]  # только верхний сегмент логарифмируем
)


RESULT:
global_rmse: 381102636.68
global_mae: 15064800.82
global_r2: 0.39
🏃 View run final_features_segmented_custom_log at: http://127.0.0.1:5000/#/experiments/2/runs/a4ed88cf0190451f915ada6103028cd2
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2
